# Task 3 - Legacy mounting

In [0]:
# Credentials needed for mounting
SCOPE       = "default2"
STORAGE     = "dlspl21databricks"
CONTAINER   = "gabrielajaniszews786"
MOUNT_POINT = "/mnt/gabrielajaniszews786_legacy"

In [0]:
client_id     = dbutils.secrets.get(SCOPE, "sp-databricks-adls-appid")
client_secret = dbutils.secrets.get(SCOPE, "sp-databricks-adls-appkey")
tenant_id     = dbutils.secrets.get(SCOPE, "tenant-id")

In [0]:
# Connection configuration
configs = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}

In [0]:
try:
    dbutils.fs.mount(
        source = f"abfss://{CONTAINER}@{STORAGE}.dfs.core.windows.net/",
        mount_point = MOUNT_POINT,
        extra_configs = configs
    )
    print(f"Successfully mounted at: {MOUNT_POINT}")
except Exception as e:
    print(f"Mounting error: {e}")

# Cluster in shared access mode blocks dbutils.fs.ount

The legacy mounting method failed on our GP1/GP2 clusters with:
Method ... DBUtilsCore.mount() is not whitelisted ... because these clusters run in SHARED access mode with Unity Catalog, which deliberately blocks mounts (they bypass UC's governance model). A real dbutils.fs.mount only works on a SINGLE USER access mode cluster.
Kept here (commented) for awareness. See the direct-SPN cell below for a version
that runs on shared clusters.

In [0]:
account_fqdn = f"{STORAGE}.dfs.core.windows.net"
for key, value in configs.items():
    spark.conf.set(f"{key}.{account_fqdn}", value)

MOUNT_POINT = f"abfss://{CONTAINER}@{account_fqdn}/"

In [0]:
display(dbutils.fs.ls(MOUNT_POINT))

Since mount() is blocked on this cluster, I tried another way to use the same SPN. Here I don't mount anything — I just give Spark the SPN login details for this storage account (client id / secret / tenant, from the secret scope) and then read the container directly with an abfss:// path.
So this is NOT a mount: MOUNT_POINT holds an abfss:// address, not a /mnt/ path. The point is just to show the Service Principal can reach the container. The ls() above lists it, which means it worked.